## Part 2.1: Initialise Profiler Run State

### Objective

Initialise the shared runtime state required by the CSV schema diagnostic profiler before loading type rules, schema registries, or source files.

Each profiler execution receives one unique session ID. The same session ID is used throughout Parts 2.1–2.6 so that all audit events, file observations, schema records, and final summary information can be traced to one profiler run.

### Dependencies

This section depends on configuration and runtime objects established earlier in the notebook:

* `MELBOURNE_TIMEZONE` from Global Runtime Settings;
* `RAW_DATA_PATH` from Part 1.2;
* `LOG_AUDIT_PATH` from Part 1.2;
* `MASTER_SCHEMA_PATH` from Part 1.2;
* `PY_SCHEMA_PATH` from Part 1.2;
* `TYPE_RULES_PATH` from Part 1.2;
* `PIPELINE_AUDIT_PATH` from Part 1.2;
* `PROFILE_SAMPLE_ROWS` from Part 1.2.

This section does not reconstruct paths or reload project configuration.

### Run State

The profiler initialises:

* `SESSION_ID` to identify the current profiler execution;
* `PROFILER_START_TIME` using Melbourne local time;
* warning and error collections;
* counters for discovered, successful, failed, and structurally different files;
* collections for file-level observations, failures, and schema comparison details.

The session ID identifies an execution only. It is separate from the approved schema version.

### Central Audit Logging

A shared `log_event()` function is created for all profiler sections.

Each log entry contains:

* a timezone-aware Melbourne timestamp;
* the profiler session ID;
* a severity level;
* an event message.

Supported severity levels are:

* `INFO`
* `WARNING`
* `ERROR`
* `CRITICAL`

The function appends every event to `pipeline_audit.log`, prints the event in the notebook, and records warnings and errors in the current run state.

### Prerequisite Validation

Before the profiler begins, this section confirms that:

* all required path variables are available;
* `RAW_DATA_PATH` exists and is a directory;
* `LOG_AUDIT_PATH` exists and is a directory;
* `PIPELINE_AUDIT_PATH` exists and is a file;
* `PROFILE_SAMPLE_ROWS` contains a valid profiling policy.

Missing global prerequisites stop execution before any schema files or CSV files are processed.

### Expected Outcome

* A unique profiler session ID is generated.
* Melbourne profiler start time is recorded.
* Shared counters and result collections are initialised.
* Central audit logging is ready for Parts 2.2–2.6.
* Profiler prerequisites are verified.
* A `SESSION_START` event is appended to `pipeline_audit.log`.


In [ ]:
# ==========================================
# Part 2.1: Initialise Profiler Run State
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4


# ------------------------------------------
# Validate dependencies from earlier parts
# ------------------------------------------

REQUIRED_RUNTIME_VARIABLES = {
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "PROFILE_SAMPLE_ROWS": PROFILE_SAMPLE_ROWS,
}


for variable_name, variable_value in REQUIRED_RUNTIME_VARIABLES.items():
    if variable_value is None and variable_name != "PROFILE_SAMPLE_ROWS":
        raise RuntimeError(
            f"Required runtime variable is unavailable: {variable_name}"
        )


# ------------------------------------------
# Validate resolved path object types
# ------------------------------------------

REQUIRED_PATH_VARIABLES = {
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "LOG_AUDIT_PATH": LOG_AUDIT_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "PY_SCHEMA_PATH": PY_SCHEMA_PATH,
    "TYPE_RULES_PATH": TYPE_RULES_PATH,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
}


for variable_name, path_value in REQUIRED_PATH_VARIABLES.items():
    if not isinstance(path_value, Path):
        raise TypeError(
            f"{variable_name} must be a pathlib.Path object."
        )


# ------------------------------------------
# Validate profiler prerequisites
# ------------------------------------------

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"RAW_DATA_PATH does not exist:\n{RAW_DATA_PATH}"
    )

if not RAW_DATA_PATH.is_dir():
    raise NotADirectoryError(
        f"RAW_DATA_PATH is not a directory:\n{RAW_DATA_PATH}"
    )


if not LOG_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"LOG_AUDIT_PATH does not exist:\n{LOG_AUDIT_PATH}"
    )

if not LOG_AUDIT_PATH.is_dir():
    raise NotADirectoryError(
        f"LOG_AUDIT_PATH is not a directory:\n{LOG_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"PIPELINE_AUDIT_PATH does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )

if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        f"PIPELINE_AUDIT_PATH is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if (
    PROFILE_SAMPLE_ROWS is not None
    and not isinstance(PROFILE_SAMPLE_ROWS, int)
):
    raise TypeError(
        "PROFILE_SAMPLE_ROWS must be an integer or null."
    )

if (
    isinstance(PROFILE_SAMPLE_ROWS, int)
    and PROFILE_SAMPLE_ROWS < 0
):
    raise ValueError(
        "PROFILE_SAMPLE_ROWS cannot be negative."
    )


# ------------------------------------------
# Initialise profiler execution identity
# ------------------------------------------

SESSION_ID = uuid4().hex[:8]

PROFILER_START_TIME = datetime.now(
    MELBOURNE_TIMEZONE
)


# ------------------------------------------
# Determine profiling policy
# ------------------------------------------

if PROFILE_SAMPLE_ROWS in (None, 0):
    PROFILING_MODE = "FULL_FILE"
    EFFECTIVE_SAMPLE_ROWS = None
else:
    PROFILING_MODE = "SAMPLED"
    EFFECTIVE_SAMPLE_ROWS = PROFILE_SAMPLE_ROWS


# ------------------------------------------
# Initialise run-level event collections
# ------------------------------------------

run_warnings = []
run_errors = []


# ------------------------------------------
# Initialise file-processing counters
# ------------------------------------------

files_discovered_count = 0
files_processed_successfully_count = 0
files_failed_count = 0
files_with_drift_count = 0
files_with_proposed_schema_variance_count = 0


# ------------------------------------------
# Initialise file-level result collections
# ------------------------------------------

discovered_files = []
processed_files = []
failed_files = []

observed_pandas_schemas_by_file = {}
observed_sql_schemas_by_file = {}

drift_details_by_file = {}
proposed_schema_variance_by_file = {}


# ------------------------------------------
# Define central profiler audit logger
# ------------------------------------------

SUPPORTED_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one profiler event to the pipeline audit log,
    print it in the notebook, and update run-level event
    collections when the event is a warning or error.
    """
    normalised_level = level.upper().strip()

    if normalised_level not in SUPPORTED_LOG_LEVELS:
        raise ValueError(
            f"Unsupported log level: {level}. "
            "Expected INFO, WARNING, ERROR, or CRITICAL."
        )

    if not isinstance(message, str) or not message.strip():
        raise ValueError(
            "Audit event message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).strftime("%Y-%m-%d %H:%M:%S %Z")

    log_entry = (
        f"{timestamp} "
        f"[RUN:{SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    # Append rather than overwrite so previous profiler
    # executions remain available for audit review.
    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(log_entry + "\n")

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "session_id": SESSION_ID,
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        run_warnings.append(event_record)

    elif normalised_level in {"ERROR", "CRITICAL"}:
        run_errors.append(event_record)

    return log_entry


# ------------------------------------------
# Record profiler session start
# ------------------------------------------

log_event(
    message=(
        "SESSION_START: CSV schema diagnostic profiler initialised. "
        f"Profiling mode={PROFILING_MODE}; "
        f"sample_rows={EFFECTIVE_SAMPLE_ROWS}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display initialisation summary
# ------------------------------------------

print("\n========== Profiler Run Initialisation ==========\n")
print(f"Session ID       : {SESSION_ID}")
print(
    "Start time       : "
    f"{PROFILER_START_TIME.isoformat(timespec='seconds')}"
)
print(f"Timezone         : {PROFILER_START_TIME.tzname()}")
print(f"Profiling mode   : {PROFILING_MODE}")
print(f"Sample rows      : {EFFECTIVE_SAMPLE_ROWS}")
print(f"Raw data path    : {RAW_DATA_PATH}")
print(f"Audit log path   : {PIPELINE_AUDIT_PATH}")

print("\n" + "=" * 55)
print("✓ Profiler run state initialised.")
print("✓ Runtime prerequisites verified.")
print("✓ Central audit logging is ready.")
print("=" * 55)